# Python Machine Learning Labs: Book Rating Predictions

### Introduction

In this project, ***.

### Data Import

In [ ]:
"""
Books ML Project
Analysis and machine learning on a dataset of books.
"""
import pandas as pd

In [ ]:
books = pd.read_csv('books.csv', on_bad_lines='warn')
print(books.columns.tolist())
print(books.shape)

In [ ]:
books.columns = books.columns.str.strip() # remove extra spaces in column names, notably num_pages

Upon importing the data with `pd.read_csv('books.csv', on_bad_lines='warn')`, the data frame is created but we know 4 rows are skipped for having an extra column. Python says these are rows 3350, 4704, 5879, and 8981. Let's see what those lines look like. 

In [ ]:
with open('books.csv', 'r', encoding='utf-8') as f:
    comma_lines = f.readlines()

# check the problematic lines (subtract 1 for 0-indexing)
for i in [3349, 4703, 5878, 8980]:
    print(f"Line {i+1}: {comma_lines[i]}")

We can see from this code the issue is due to an extra comma in the author column. It's much easier to see this issue when viewing the CSV in Excel, filtering this erroneous 13th column by all non-blank values to see these 4 rows. We can fix these rows manually and add them to our dataframe if we don't want to lose the data. 

In [ ]:
bad_lines = [3349, 4703, 5878, 8980]
manual_rows = []
with open('books.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i in bad_lines:
        line = lines[i].strip()
        parts = line.split(',', 12)
        if len(parts) == 13:
            parts = [parts[0], parts[1], parts[2] + parts[3], parts[4], parts[5], parts[6], parts[7], parts[8], parts[9], parts[10], parts[11], parts[12]]
        manual_rows.append(parts)

manual_df = pd.DataFrame(manual_rows, columns=books.columns)
manual_df # ensure it has the correct data

In [ ]:
# We confirmed the data is correct, so we can concatonate manual_df to books.
books = pd.concat([books, manual_df], ignore_index=True)
print(books.shape) # Print shape to confirm it is the expected 11127 rows, 12 columns.

### Data Exploration & Cleaning

We begin exploring the data to get a better idea of how it's structured. 

In [ ]:
books.head() # General view of the data.

In [ ]:
books.info()
books.describe()

If we look to our output from `books.describe()`, we can learn several things. For example, we see the datatypes are objects or strings, we may want to convert these to their proper types, like dates to datetime. The machine learning model will not use some features for the predictions, such as the bookID, isbn, and isbn13. Therefore, these columns can be removed. 

In [ ]:
books_less_columns = books.copy()
books_less_columns = books_less_columns.drop(columns=['bookID', 'isbn', 'isbn13'])

In [ ]:
books["publication_date"].isna().sum()

In [ ]:
books_dates = books_less_columns.copy()
books_dates["publication_date"] = pd.to_datetime(
    books_dates["publication_date"],
    errors="coerce"
    )
books_dates["publication_date"].isna().sum() # Check how many publication dates are invalid.

The change of the publication date column to datetime caused two rows to be invalid, let's view those original rows to see why.

In [ ]:
books[
    pd.to_datetime(
        books["publication_date"],
        errors="coerce"
    ).isna()
][["title", "publication_date"]]

It's because these aren't real dates, there are not 31 days in November or June. We could seek this data and manually fix it, but for consistency and reproducibility of the project they can be left as missing, since 2 rows of 11,000 won't make a meaningful difference. We will remove those 2 rows now. Additionally, we will add a column for just the year, since the month and day are less relevant to the model. As an additional step, we will calculate the age of the book to make the number more intuitive for the model instead of a larger number for year, the smaller "book_age" will have the same effect on the model. 

In [ ]:
books_dates = books_dates.dropna(subset=["publication_date"])
books_dates["publication_year"] = books_dates["publication_date"].dt.year
books_dates["book_age"] = 2026 - books_dates["publication_year"]
books_dates[["publication_year", "book_age"]] = books_dates[["publication_year", "book_age"]].astype(int)

In [ ]:
books_dates[["publication_date", "publication_year", "book_age"]].info() # Confirm the new columns are now the correct types and that there are no null values in those columns.

Now that we have fixed the dates, we can move along with the cleaning. Next, let's continue converting the objects to numeric types where appropriate. 

In [ ]:
books_numeric = books_dates.copy()
books_numeric.dtypes

We want to convert these columns `"average_rating", "num_pages", "ratings_count", "text_reviews_count"` into numeric type, but first we want to ensure there is not missing data to start and that at the end we get the same data out, just as `int64` or `float64` instead of `object`.

In [ ]:
cols = ["average_rating", "num_pages", "ratings_count", "text_reviews_count"]

for c in cols:
    coerced = pd.to_numeric(books_numeric[c], errors="coerce")
    
    print("\n", c)
    print("original nulls:", books_numeric[c].isna().sum())
    print("new nulls after coercion:", coerced.isna().sum())

We have no nulls to begin with and none after the conversion, so it's likely safe to convert. 

In [ ]:
books_numeric[cols] = books_numeric[cols].apply(pd.to_numeric)

In [ ]:
books_numeric[cols].dtypes # Checking the dtypes of the numeric columns to ensure they are now numeric types.

In [ ]:
books_numeric[cols].describe() # Checking the descriptive statistics of the numeric columns to ensure they are now numeric types and to get a sense of the data distribution.

With all of our data types corrected, we can now move our attention to the language code. 

In [ ]:
books_lang = books_numeric.copy()
books_lang["language_code"].unique()

In [ ]:
# First we can standardize the books in English to all be "eng".
books_lang["language_code"] = (
    books_lang["language_code"]
    .str.lower()
    .replace({
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)
books_lang["language_code"].value_counts()

There are many languages that have very few appearances. Therefore, we can keep the bigger categories, and put the smaller ones into a grouped "other" category. What constitutes bigger? For now, we will keep Japanese as the smallest category with 46, but even that is very few compared to the others. 

In [ ]:
top_langs = ["eng", "spa", "fre", "ger", "jpn"]

books_lang["language_code"] = books_lang["language_code"].apply(
    lambda x: x if x in top_langs else "other"
)

books_lang["language_code"].value_counts()

The languages are now cleaned up and ready for one-hot encoding later. Let's continue with other various cleaning steps. 

In [ ]:
books_clean = books_lang.copy()
books_clean.isna().sum() # Ensuring no missing values in the dataset before proceeding.

In [ ]:
books_clean["average_rating"].describe() # Investigating the average_rating column to see if there are any 0 values that should be removed from the dataset.

A rating of 0 should not be possible, but we have a minimum of 0. Let's see how many and try to figure out why. 

In [ ]:
books_clean[books_clean["average_rating"] == 0].shape[0] # Getting a count of how many books have an average rating of 0.

In [ ]:
books_clean[books_clean["average_rating"] == 0] # Viewing the books that have an average rating of 0 to see if they are valid or if they should be removed from the dataset.

The books with a rating of 0 don't have any reviews, so the 0 represents no data. Our ML model will see the 0 and think it is a valid rating, which is not true. To avoid affecting the model, these will be coverted to NaN and dropped. From `books_clean["average_rating"].describe()` we know there are 11125 rows, and from `books_clean[books_clean["average_rating"] == 0].shape[0]` we know there are 26 books with a rating of 0. Once dropped, our dataset should have 11099 rows. 

In [ ]:
import numpy as np
books_clean.loc[books_clean["average_rating"] == 0, "average_rating"] = np.nan
books_clean = books_clean.dropna(subset=["average_rating"])

In [ ]:
books_clean["average_rating"].describe() # Verifying the minimum is now 1, and the count is 11099 as expected.

Next we will ensure there are no duplicates. 

In [ ]:
books_clean.duplicated().sum() # Checking for duplicate rows in the dataset.

There are no duplicated entire rows, however we should check for duplicate titles in the dataset. This is an important distinction from the last step because the books.describe() at the start showed a difference in the number of book titles and unique titles.

In [ ]:
print(books_clean["title"].duplicated().sum())
print(books_clean.duplicated(subset=["title", "authors"]).sum()) # Checking for duplicate books based on title and author, as there may be different editions of the same book.

The select key uses both `title` and `authors` because title alone is not a unique enough identifier. Multiple books can share the same title, but the combination of title and author more reliably represents a unique book entity, reducing false positives in duplicate removal. This can be seen with the different values in the prior block of code. 

In [ ]:
books_clean[books_clean.duplicated(subset=["title", "authors"], keep=False)].sort_values(["title", "authors"]) # With keep=False we also see the first occurrence of the duplicate rows, which allows us to see all the duplicates in the dataset.

We see many books have multiple entries with varying metadata. We don't want the model to run over these several times and skew the results with bias from duplicates, so we need to choose one entry for each of these to keep. Under the assumption that the most-rated entry is most representative of the book, that is the entry that will be kept. 

In [ ]:
books_clean = (
    books_clean
    .sort_values("ratings_count", ascending=False)
    .drop_duplicates(subset=["title", "authors"], keep="first")
)

In [ ]:
print(books_clean["title"].duplicated().sum()) # There are still some duplicates, but they could be different editions of the same book, so we will keep them for now. The prior step removed most of the duplicates.
print(books_clean.duplicated(subset=["title", "authors"]).sum()) # Ensuring this is now 0, as we have removed the duplicates based on title and author, keeping the most popular edition of each book.

Next we want to do some general cleanup, like removing any extra spaces in the data, and ensuring consistency in formatting. 

In [ ]:
books_clean["title"] = (
    books_clean["title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
books_clean["authors"] = (
    books_clean["authors"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)
books_clean["publisher"] = (
    books_clean["publisher"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)
books_clean["language_code"] = books_clean["language_code"].astype("category")

Next we take a look at a summary of some of the numerical data to view any patterns or anything strange.

In [ ]:
books_clean[["num_pages", "ratings_count", "text_reviews_count"]].describe()

We notice the minimum value for all three is 0, which is fine for text reviews since it is optional. However, we should not have books with 0 ratings as that would imply they have a rating of 0, but those rows were already dropped. Let's investigate that further before we address the number of pages. 

In [ ]:
books_clean[
    (books_clean["average_rating"] > 0) &
    (books_clean["ratings_count"] == 0)
].shape[0] # Checking for books that have an average rating greater than 0 but have no ratings count, which is not possible. We will remove these books from the dataset.

In [ ]:
books_clean = books_clean[~(
        (books_clean["average_rating"] > 0) &
        (books_clean["ratings_count"] == 0)
    )]

In [ ]:
books_clean[
    (books_clean["average_rating"] > 0) &
    (books_clean["ratings_count"] == 0)
].shape[0] # Verify that there are no books with an average rating greater than 0 but have no ratings count.

Now we will address the number of pages. 

In [ ]:
books_clean[books_clean["num_pages"] == 0].shape[0] # Checking for books that have 0 pages, which is not possible. We will remove these books from the dataset.

In [ ]:
zero_pages = books_clean[books_clean["num_pages"] == 0]

zero_pages[[
    "title",
    "authors",
    "num_pages",
    "average_rating",
    "ratings_count",
    "text_reviews_count",
    "publication_year",
    "publisher",
]].head(20) # Viewing the books that have 0 pages to see if they are valid or if they should be removed from the dataset.

These books have only bad data for the number of pages, but otherwise look okay. We can change the 0 to NaN and then replace those with the median of the dataset to impute those values. 

In [ ]:
books_clean.loc[
    books_clean["num_pages"] == 0,
    "num_pages"
] = np.nan

books_clean["num_pages"] = books_clean["num_pages"].fillna(
    books_clean["num_pages"].median())

Then we can view the numerical data to see if we should log adjust to help with outliers and large variety in values due to popularity of certain books. 

In [ ]:
books_clean[["num_pages", "ratings_count", "text_reviews_count"]].describe() # Once again viewing the descriptive statistics after making changes.

We see now that instead of 0 pages, we still have books with 1 page. This could be true if they are pamphlets in the dataset, but it's impossible to know and unjust to simply remove those rows or impute data. Let's see how many books have under 10 pages and see how many there are to see how they may affect the data.

In [ ]:
books_clean[books_clean["num_pages"] <= 10].sort_values("num_pages").shape[0]

For our dataset size, it is not a big deal, but lets address it while also addressing the spread of values overall via log transformation. This way, our skew will be minimal from the very high values of each row as well as the low counts, such as the books with less than 10 pages.

In [ ]:
books_clean["ratings_count_log"] = np.log1p(
    books_clean["ratings_count"])
books_clean["num_pages_log"] = np.log1p(
    books_clean["num_pages"])
books_clean["text_reviews_count_log"] = np.log1p(
    books_clean["text_reviews_count"])

In [ ]:
books_clean.head() # Viewing the cleaned dataset to ensure it looks correct after removing duplicates and cleaning the data.

### That's all for the data cleaning and preprocessing, and some feature engineering was done. Let's get into the big feature engineering now, particularly in preparation for ML. 

In [ ]:
books_ML = books_clean.copy() # Creating a copy of the cleaned dataset to use for machine learning.

First step will be to one-hot encode the language code to make it more readable for the ML algorithms. 

In [ ]:
books_ML["language_code"].value_counts()

In [ ]:
books_ML = pd.get_dummies(
    books_ML,
    columns=["language_code"],
    drop_first=True # If all are 0, then the book is in English, so we can drop the first column to avoid multicollinearity.
)

In [ ]:
books_ML.head() # Viewing the dataset to ensure the one-hot encoding of the language_code column was successful.

What should we one-hot encode? Lang-code is easy to say yes, but what about authors and publisher? It would make a huge matrix of mostly zeroes, and is unlikely to be the best option. 
Also, remove the title since we are using the features of the books to predict rating? We can't really use advanced setups to try and get cues from the names, like genre cues or tone or anything. 